<a href="https://colab.research.google.com/github/Ghhaidaa/Data-Engineering-for-AI-System-DAICO/blob/main/notebooks/SDAIA_Books_Platform_v5_FIXED.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **SDAIA** **Books** **Platform**

A data engineering platform for validating, streaming, and storing book registration data submitted by publishers across Saudi Arabia.

# Data Ingestion and Delta Lake

This notebook presents the implementation of the **Data Ingestion** and **Delta Lakehouse** components for the SDAIA Books Platform.

The implementation includes:

- Data validation using **Pydantic**.
- Real-time data streaming with **Apache Kafka** (Producer & Consumer).
- Delta Lake **Bronze**, **Silver**, and **Gold** layers.
- **MERGE (Upsert)** using `book_id` as the business key.
- **Schema Enforcement** demonstration to validate schema consistency.

This notebook corresponds to **Parts 1 & 2** of the project deliverables.

In [1]:
!apt-get update -qq

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)


In [2]:
!apt-get install -y openjdk-17-jdk-headless

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
openjdk-17-jdk-headless is already the newest version (17.0.19+10-1~22.04.2).
0 upgraded, 0 newly installed, 0 to remove and 118 not upgraded.


In [3]:
!pip install kafka-python pydantic faker

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 614.1/614.1 kB 31.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 63.4 MB/s eta 0:00:00


In [4]:
import pandas as pd
import random
from faker import Faker

fake = Faker()

# Make results reproducible
Faker.seed(42)
random.seed(42)

In [5]:
publisher_names = [
    "Jarir Publishing",
    "Obeikan Publishing",
    "King Fahad National Library",
    "Dar Al-Minhaj",
    "Saudi Research Publishing"
]

publisher_cities = [
    "Riyadh",
    "Jeddah",
    "Dammam",
    "Madinah",
    "Abha"
]

categories = [
    "Artificial Intelligence",
    "Data Science",
    "Cybersecurity",
    "Education",
    "Government",
    "Culture",
    "History"
]

languages = ["Arabic", "English"]

records = []

for book_id in range(1, 101):

    isbn = f"978603{random.randint(1000000,9999999)}"

    records.append({
        "book_id": book_id,
        "isbn": isbn,
        "title": fake.sentence(nb_words=4).replace(".", ""),
        "author": fake.name(),
        "publisher_name": random.choice(publisher_names),
        "publisher_city": random.choice(publisher_cities),
        "category": random.choice(categories),
        "language": random.choice(languages),
        "publication_year": random.randint(2015, 2026),
        "submission_date": fake.date_this_year().strftime("%Y-%m-%d")
    })

df = pd.DataFrame(records)

df.head()

,book_id,isbn,title,author,publisher_name,publisher_city,category,language,publication_year,submission_date
0,1,9786032867825,Agent every,Noah Rhodes,Jarir Publishing,Dammam,Data Science,Arabic,2017,2026-07-01
1,2,9786032719583,Opportunity all,David Guzman,Saudi Research Publishing,Riyadh,Government,English,2015,2026-04-24
2,3,9786031499914,Information last everything thank serve,Jay Ramirez,Jarir Publishing,Jeddah,Data Science,Arabic,2023,2026-02-13
3,4,9786034335942,Bill here grow gas,Andrew Stevens,Saudi Research Publishing,Madinah,Data Science,English,2024,2026-01-26
4,5,9786035667265,Bad fall pick those,Corey Jones,Jarir Publishing,Jeddah,Culture,English,2020,2026-05-15


In [6]:
# Missing ISBN
df.loc[0, "isbn"] = None

# Empty title
df.loc[1, "title"] = ""

# Unsupported language
df.loc[2, "language"] = "French"

# Future publication year
df.loc[3, "publication_year"] = 2050

# Invalid date format
df.loc[4, "submission_date"] = "2026/15/40"

df.head()

,book_id,isbn,title,author,publisher_name,publisher_city,category,language,publication_year,submission_date
0,1,None,Agent every,Noah Rhodes,Jarir Publishing,Dammam,Data Science,Arabic,2017,2026-07-01
1,2,9786032719583,,David Guzman,Saudi Research Publishing,Riyadh,Government,English,2015,2026-04-24
2,3,9786031499914,Information last everything thank serve,Jay Ramirez,Jarir Publishing,Jeddah,Data Science,French,2023,2026-02-13
3,4,9786034335942,Bill here grow gas,Andrew Stevens,Saudi Research Publishing,Madinah,Data Science,English,2050,2026-01-26
4,5,9786035667265,Bad fall pick those,Corey Jones,Jarir Publishing,Jeddah,Culture,English,2020,2026/15/40


In [7]:
df.to_csv("books_data.csv", index=False)

print("Dataset created successfully!")
print(f"Total records: {len(df)}")

Dataset created successfully!
Total records: 100


In [8]:
from pydantic import BaseModel, Field, ValidationError, field_validator
from datetime import datetime

In [9]:
class BookRecord(BaseModel):
    book_id: int = Field(gt=0)
    isbn: str
    title: str
    author: str
    publisher_name: str
    publisher_city: str
    category: str
    language: str
    publication_year: int
    submission_date: str

    @field_validator("isbn")
    @classmethod
    def validate_isbn(cls, value):
        if not value:
            raise ValueError("ISBN cannot be empty.")
        return value

    @field_validator("title")
    @classmethod
    def validate_title(cls, value):
        if not value.strip():
            raise ValueError("Title cannot be empty.")
        return value

    @field_validator("language")
    @classmethod
    def validate_language(cls, value):
        if value not in ["Arabic", "English"]:
            raise ValueError("Language must be Arabic or English.")
        return value

    @field_validator("publication_year")
    @classmethod
    def validate_year(cls, value):
        current_year = datetime.now().year
        if value > current_year:
            raise ValueError("Publication year cannot be in the future.")
        return value

    @field_validator("submission_date")
    @classmethod
    def validate_date(cls, value):
        datetime.strptime(value, "%Y-%m-%d")
        return value

In [10]:
valid_records = []
invalid_records = []

for _, row in df.iterrows():
    try:
        record = BookRecord(**row.to_dict())
        valid_records.append(record.model_dump())
    except ValidationError as e:
        invalid_records.append({
            **row.to_dict(),
            "error": str(e)
        })

valid_df = pd.DataFrame(valid_records)
invalid_df = pd.DataFrame(invalid_records)

print(f"✅ Valid Records: {len(valid_df)}")
print(f"❌ Invalid Records: {len(invalid_df)}")

✅ Valid Records: 95
❌ Invalid Records: 5


In [11]:
valid_df.to_csv("valid_books.csv", index=False)
invalid_df.to_csv("rejected_books.csv", index=False)

print("Files saved successfully!")

print(f"Valid records: {len(valid_df)}")
print(f"Invalid records: {len(invalid_df)}")

Files saved successfully!
Valid records: 95
Invalid records: 5


In [12]:
# Download and extract Kafka (single clean setup - no duplicate download/format)
!wget -q https://archive.apache.org/dist/kafka/3.7.0/kafka_2.13-3.7.0.tgz
!tar -xzf kafka_2.13-3.7.0.tgz


In [13]:
%cd kafka_2.13-3.7.0


/content/kafka_2.13-3.7.0


In [14]:
import os

# Make sure there is no leftover storage from a previous run - this is what
# previously caused: "Invalid cluster.id ... Expected X, but read Y"
!rm -rf /tmp/kraft-combined-logs

cluster_id = os.popen("bin/kafka-storage.sh random-uuid").read().strip()
print(cluster_id)

!bin/kafka-storage.sh format -t $cluster_id -c config/kraft/server.properties


Ztpfv9eqSyKm3ReDGi-NEw
metaPropertiesEnsemble=MetaPropertiesEnsemble(metadataLogDir=Optional.empty, dirs={/tmp/kraft-combined-logs: EMPTY})
Formatting /tmp/kraft-combined-logs with metadata.version 3.7-IV4.


In [15]:
import subprocess
import time

# Start the Kafka broker (started only ONCE - starting it twice would try to
# bind the same port and is unnecessary)
broker = subprocess.Popen(
    ["bin/kafka-server-start.sh", "config/kraft/server.properties"]
)

time.sleep(10)

print("Kafka Broker started!")


Kafka Broker started!


In [16]:
!bin/kafka-topics.sh \
  --create \
  --topic books \
  --bootstrap-server localhost:9092

Created topic books.


In [17]:
from kafka import KafkaProducer
import pandas as pd
import json


# Step 1: Create a Kafka Producer
producer = KafkaProducer(
    bootstrap_servers="localhost:9092",
    value_serializer=lambda x: json.dumps(x).encode("utf-8")
)

# Step 2: Load the validated dataset
valid_books = pd.read_csv("/content/valid_books.csv")


# Step 3: Send each record to the Kafka topic
for _, record in valid_books.iterrows():
    producer.send("books", value=record.to_dict())


# Step 4: Ensure all messages are delivered
producer.flush()

print(f"Successfully sent {len(valid_books)} records to the 'books' topic.")

Successfully sent 95 records to the 'books' topic.


In [18]:
from kafka import KafkaConsumer
import json


# Step 1: Create a Kafka Consumer
# consumer_timeout_ms stops iteration automatically if no new message arrives
# for 10 seconds, instead of hanging forever if the expected count is wrong.
consumer = KafkaConsumer(
    "books",
    bootstrap_servers="localhost:9092",
    auto_offset_reset="earliest",
    enable_auto_commit=True,
    consumer_timeout_ms=10000,
    value_deserializer=lambda x: json.loads(x.decode("utf-8"))
)

print("Reading records from Kafka...\n")

# Step 2: Read records from Kafka
# Stop after reading exactly as many records as were produced (dynamic,
# not hard-coded), so this works even if the number of valid records changes.
expected_count = len(valid_books)
count = 0

for message in consumer:
    print(message.value)
    count += 1

    if count == expected_count:
        break

consumer.close()

print(f"\nSuccessfully consumed {count} / {expected_count} records.")
if count < expected_count:
    print("Warning: consumer timed out before reaching the expected count.")


/tmp/ipykernel_454/170633921.py:8: DeprecationWarning: value_deserializer does not implement kafka.serializer.Deserializer
  consumer = KafkaConsumer(


Reading records from Kafka...

{'book_id': 6, 'isbn': 9786035661907, 'title': 'World talk term', 'author': 'Jesse Flowers', 'publisher_name': 'Obeikan Publishing', 'publisher_city': 'Jeddah', 'category': 'History', 'language': 'English', 'publication_year': 2016, 'submission_date': '2026-02-03'}
{'book_id': 7, 'isbn': 9786032556017, 'title': 'Decide environment view possible', 'author': 'Carolyn Daniel', 'publisher_name': 'Dar Al-Minhaj', 'publisher_city': 'Riyadh', 'category': 'Cybersecurity', 'language': 'English', 'publication_year': 2024, 'submission_date': '2026-02-03'}
{'book_id': 8, 'isbn': 9786035437923, 'title': 'Establish understand read detail', 'author': 'Randy Brown', 'publisher_name': 'Jarir Publishing', 'publisher_city': 'Madinah', 'category': 'Government', 'language': 'Arabic', 'publication_year': 2021, 'submission_date': '2026-06-16'}
{'book_id': 9, 'isbn': 9786032322047, 'title': 'Husband at tree note', 'author': 'Christy Porter', 'publisher_name': 'Saudi Research Pub

In [19]:
!pip install -q pyspark==3.5.0 delta-spark==3.2.0

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 316.9/316.9 MB 5.2 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 200.5/200.5 kB 17.4 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dataproc-spark-connect 1.1.0 requires pyspark[connect]~=4.0.0, but you have pyspark 3.5.0 which is incompatible.


In [20]:
from pyspark.sql import SparkSession
from delta import configure_spark_with_delta_pip

builder = (
    SparkSession.builder
    .appName("SDAIA Books Pipeline")
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
)

spark = configure_spark_with_delta_pip(builder).getOrCreate()

print("Spark Session is ready!")

Spark Session is ready!


In [21]:
from pyspark.sql.functions import current_timestamp


# Step 1: Load the validated dataset
bronze_df = spark.read.csv(
    "/content/valid_books.csv",
    header=True,
    inferSchema=True
)


# Step 2: Add ingestion timestamp
bronze_df = bronze_df.withColumn(
    "ingestion_time",
    current_timestamp()
)


# Step 3: Save as Delta Bronze table
bronze_df.write \
    .format("delta") \
    .mode("overwrite") \
    .save("/content/delta/bronze")

print("Bronze layer created successfully!")

Bronze layer created successfully!


In [22]:
bronze = spark.read.format("delta").load("/content/delta/bronze")

print(f"Number of records: {bronze.count()}")

bronze.show(5, truncate=False)

Number of records: 95
+-------+-------------+--------------------------------+--------------+-------------------------+--------------+-------------+--------+----------------+---------------+--------------------------+
|book_id|isbn         |title                           |author        |publisher_name           |publisher_city|category     |language|publication_year|submission_date|ingestion_time            |
+-------+-------------+--------------------------------+--------------+-------------------------+--------------+-------------+--------+----------------+---------------+--------------------------+
|6      |9786035661907|World talk term                 |Jesse Flowers |Obeikan Publishing       |Jeddah        |History      |English |2016            |2026-02-03     |2026-07-23 02:01:10.946813|
|7      |9786032556017|Decide environment view possible|Carolyn Daniel|Dar Al-Minhaj            |Riyadh        |Cybersecurity|English |2024            |2026-02-03     |2026-07-23 02:01:10.946813

In [23]:
from pyspark.sql.functions import col


# Step 1: Read Bronze layer
silver_df = spark.read.format("delta").load("/content/delta/bronze")


# Step 2: Clean the data
silver_df = (
    silver_df
    .dropDuplicates()
    .withColumn("publication_year", col("publication_year").cast("int"))
    .withColumn("submission_date", col("submission_date").cast("date"))
)


# Step 3: Save as Delta Silver table
silver_df.write \
    .format("delta") \
    .mode("overwrite") \
    .save("/content/delta/silver")

print("Silver layer created successfully!")

Silver layer created successfully!


In [24]:
silver = spark.read.format("delta").load("/content/delta/silver")

print(f"Number of records: {silver.count()}")

silver.show(5, truncate=False)

Number of records: 95
+-------+-------------+-------------------------------------+-----------------+---------------------------+--------------+-------------+--------+----------------+---------------+--------------------------+
|book_id|isbn         |title                                |author           |publisher_name             |publisher_city|category     |language|publication_year|submission_date|ingestion_time            |
+-------+-------------+-------------------------------------+-----------------+---------------------------+--------------+-------------+--------+----------------+---------------+--------------------------+
|87     |9786034940441|Matter management ball               |Jeffrey Morris   |King Fahad National Library|Jeddah        |History      |Arabic  |2017            |2026-02-20     |2026-07-23 02:01:10.946813|
|25     |9786035418934|Better present music address behavior|Cynthia Russell  |Saudi Research Publishing  |Jeddah        |Government   |Arabic  |2025     

In [25]:
from pyspark.sql.functions import count


# Step 1: Read Silver layer
silver = spark.read.format("delta").load("/content/delta/silver")


# Step 2: Create Gold layer
# Count books by category and language
gold_df = (
    silver
    .groupBy("category", "language")
    .agg(count("*").alias("number_of_books"))
)


# Step 3: Save as Delta Gold table
gold_df.write \
    .format("delta") \
    .mode("overwrite") \
    .save("/content/delta/gold")

print("Gold layer created successfully!")

Gold layer created successfully!


In [26]:
gold = spark.read.format("delta").load("/content/delta/gold")

gold.orderBy("category", "language").show(truncate=False)

+-----------------------+--------+---------------+
|category               |language|number_of_books|
+-----------------------+--------+---------------+
|Artificial Intelligence|Arabic  |8              |
|Artificial Intelligence|English |5              |
|Culture                |Arabic  |9              |
|Culture                |English |5              |
|Cybersecurity          |Arabic  |3              |
|Cybersecurity          |English |7              |
|Data Science           |Arabic  |5              |
|Data Science           |English |10             |
|Education              |Arabic  |8              |
|Education              |English |7              |
|Government             |Arabic  |8              |
|Government             |English |5              |
|History                |Arabic  |6              |
|History                |English |9              |
+-----------------------+--------+---------------+



In [27]:
from pyspark.sql import Row
from pyspark.sql.functions import current_timestamp, to_date

# Read the schema from Silver
silver = spark.read.format("delta").load("/content/delta/silver")

updates = [
    (
        46,
        "9786035449368",
        "Updated Book Title",
        "Michelle Evans",
        "Dar Al-Minhaj",
        "Dammam",
        "Education",
        "English",
        2025,
        "2026-07-22"
    )
]

updates_df = spark.createDataFrame(
    updates,
    schema="""
    book_id INT,
    isbn STRING,
    title STRING,
    author STRING,
    publisher_name STRING,
    publisher_city STRING,
    category STRING,
    language STRING,
    publication_year INT,
    submission_date STRING
    """
)

updates_df = (
    updates_df
    .withColumn("submission_date", to_date("submission_date"))
    .withColumn("ingestion_time", current_timestamp())
)

updates_df.show()

+-------+-------------+------------------+--------------+--------------+--------------+---------+--------+----------------+---------------+--------------------+
|book_id|         isbn|             title|        author|publisher_name|publisher_city| category|language|publication_year|submission_date|      ingestion_time|
+-------+-------------+------------------+--------------+--------------+--------------+---------+--------+----------------+---------------+--------------------+
|     46|9786035449368|Updated Book Title|Michelle Evans| Dar Al-Minhaj|        Dammam|Education| English|            2025|     2026-07-22|2026-07-23 02:02:...|
+-------+-------------+------------------+--------------+--------------+--------------+---------+--------+----------------+---------------+--------------------+



MERGE

The following example demonstrates a Delta Lake MERGE (Upsert) operation using `book_id` as the business key.

In [28]:
from delta.tables import DeltaTable

silver_table = DeltaTable.forPath(spark, "/content/delta/silver")

(
    silver_table.alias("target")
    .merge(
        updates_df.alias("source"),
        "target.book_id = source.book_id"
    )
    .whenMatchedUpdateAll()
    .whenNotMatchedInsertAll()
    .execute()
)

print("Merge completed successfully!")

Merge completed successfully!


In [29]:
silver = spark.read.format("delta").load("/content/delta/silver")

silver.filter("book_id = 46").show(truncate=False)

+-------+-------------+------------------+--------------+--------------+--------------+---------+--------+----------------+---------------+--------------------------+
|book_id|isbn         |title             |author        |publisher_name|publisher_city|category |language|publication_year|submission_date|ingestion_time            |
+-------+-------------+------------------+--------------+--------------+--------------+---------+--------+----------------+---------------+--------------------------+
|46     |9786035449368|Updated Book Title|Michelle Evans|Dar Al-Minhaj |Dammam        |Education|English |2025            |2026-07-22     |2026-07-23 02:02:09.880961|
+-------+-------------+------------------+--------------+--------------+--------------+---------+--------+----------------+---------------+--------------------------+



## Schema Enforcement
The following example demonstrates Delta Lake schema enforcement by attempting to write data that does not match the existing table schema.

In [30]:
from pyspark.sql import Row

# Create invalid data with an extra column
invalid_data = [
    Row(
        book_id=999,
        isbn="9786030000000",
        title="Invalid Book",
        author="Test Author",
        publisher_name="Test Publisher",
        publisher_city="Riyadh",
        category="Education",
        language="English",
        publication_year=2025,
        submission_date="2026-07-22",
        extra_column="This column should not exist"
    )
]

invalid_df = spark.createDataFrame(invalid_data)

try:
    invalid_df.write \
        .format("delta") \
        .mode("append") \
        .save("/content/delta/silver")

    print("Schema accepted.")

except Exception as e:
    print("Schema Enforcement is working!")
    print(e)

Schema Enforcement is working!
[DELTA_FAILED_TO_MERGE_FIELDS] Failed to merge fields 'book_id' and 'book_id'


# Stage 4-RAG Pipeline

This notebook presents the implementation of the Retrieval-Augmented Generation (RAG) component for the SDAIA Books Platform.

The implementation includes:

- Preparing structured book data from the Gold layer.
- Building source documents and splitting them into smaller chunks.
- Generating embeddings for each chunk.
- Storing embeddings in ChromaDB.
- Implementing hybrid retrieval using dense search and BM25.
- Combining retrieval results using Reciprocal Rank Fusion.
- Reranking retrieved chunks using a CrossEncoder model.
- Generating a grounded answer with a citation to the retrieved source.

In [31]:
!pip install chromadb sentence-transformers rank-bm25

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 55.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 21.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 107.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.7/18.7 MB 65.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.2/137.2 kB 12.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 204.6/204.6 kB 15.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.7/61.7 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 5.0 MB/s eta 0:00:00
  Attempting uninstall: opentelemetry-api
    Found

In [32]:
# Import Libraries

import numpy as np
import pandas as pd
import chromadb

from sentence_transformers import SentenceTransformer, CrossEncoder
from rank_bm25 import BM25Okapi

In [33]:
# Step 1: Read Gold Layer

from pyspark.sql import SparkSession

spark = SparkSession.builder.getOrCreate()

gold_df = spark.read.format("delta").load("/content/delta/gold")

gold_df.show(5, truncate=False)

+-----------------------+--------+---------------+
|category               |language|number_of_books|
+-----------------------+--------+---------------+
|Data Science           |English |10             |
|Cybersecurity          |English |7              |
|Artificial Intelligence|Arabic  |8              |
|History                |Arabic  |6              |
|Culture                |Arabic  |9              |
+-----------------------+--------+---------------+
only showing top 5 rows



In [34]:
# Step 2: Convert Gold Data to Documents (with metadata for citations)

documents = []       # full text per category (before chunking)
doc_metadata = []    # source info for each document, used later for citations

for i, row in enumerate(gold_df.collect()):
    doc_id = f"doc_{i}"
    text = (
        f"In the {row['category']} category, the Smart Library holds "
        f"{row['number_of_books']} books, most of them written in {row['language']}. "
        f"This reflects the demand and popularity of {row['category']} titles "
        f"among readers who prefer {row['language']} content. "
        f"Librarians use this data to decide which {row['category']} titles "
        f"to restock in {row['language']} for the upcoming semester."
    )
    documents.append(text)
    doc_metadata.append({
        "doc_id": doc_id,
        "category": row["category"],
        "language": row["language"],
        "number_of_books": row["number_of_books"]
    })

print(f"Built {len(documents)} source documents.")
print(documents[0])


Built 14 source documents.
In the Data Science category, the Smart Library holds 10 books, most of them written in English. This reflects the demand and popularity of Data Science titles among readers who prefer English content. Librarians use this data to decide which Data Science titles to restock in English for the upcoming semester.


In [35]:
# Step 2b: Chunk Documents into Passages
# Real RAG pipelines split long documents into overlapping chunks before
# embedding, so retrieval can return precise, relevant passages instead of
# whole documents. Here we chunk by words with a small overlap between chunks.

def chunk_text(text, chunk_size=15, overlap=5):
    words = text.split()
    chunks = []
    start = 0
    while start < len(words):
        end = start + chunk_size
        chunk = " ".join(words[start:end])
        chunks.append(chunk)
        if end >= len(words):
            break
        start += (chunk_size - overlap)
    return chunks

chunk_texts = []      # text of every chunk, used for embeddings + BM25
chunk_ids = []        # unique id per chunk
chunk_sources = []    # which source document each chunk came from (for citations)

for doc, meta in zip(documents, doc_metadata):
    doc_chunks = chunk_text(doc)
    for c_idx, chunk in enumerate(doc_chunks):
        chunk_texts.append(chunk)
        chunk_ids.append(f"{meta['doc_id']}_chunk_{c_idx}")
        chunk_sources.append(meta)

print(f"Split {len(documents)} documents into {len(chunk_texts)} chunks.")
print("Example chunk:", chunk_texts[0])
print("Example chunk id:", chunk_ids[0])


Split 14 documents into 70 chunks.
Example chunk: In the Data Science category, the Smart Library holds 10 books, most of them written
Example chunk id: doc_0_chunk_0


In [36]:
# Step 3: Load Embedding Model

embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

print("Embedding model loaded successfully!")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding model loaded successfully!


In [37]:
# Step 4: Generate Embeddings for each chunk

embeddings = embedding_model.encode(chunk_texts).tolist()

print(f"Generated {len(embeddings)} embeddings (one per chunk).")


Generated 70 embeddings (one per chunk).


In [38]:
# Step 5: Create ChromaDB Collection (store chunks, not whole documents)

client = chromadb.Client()

collection = client.create_collection(name="books")

collection.add(
    documents=chunk_texts,
    embeddings=embeddings,
    ids=chunk_ids,
    metadatas=chunk_sources
)

print("Chunks stored in ChromaDB successfully!")


Chunks stored in ChromaDB successfully!


In [39]:
# Step 6: Build BM25 Index over chunks

tokenized_documents = [chunk.split() for chunk in chunk_texts]

bm25 = BM25Okapi(tokenized_documents)

print("BM25 index created successfully over chunks!")


BM25 index created successfully over chunks!


In [40]:
# Step 7: User Query

query = "How many Arabic Data Science books are available?"

print(query)

How many Arabic Data Science books are available?


In [41]:
# Step 8: Vector Search

query_embedding = embedding_model.encode(query).tolist()

vector_results = collection.query(
    query_embeddings=[query_embedding],
    n_results=5
)

print(vector_results["documents"][0])

['books, most of them written in Arabic. This reflects the demand and popularity of Data', 'prefer Arabic content. Librarians use this data to decide which Data Science titles to restock', 'demand and popularity of Data Science titles among readers who prefer Arabic content. Librarians use', 'Data Science titles to restock in Arabic for the upcoming semester.', 'prefer Arabic content. Librarians use this data to decide which Artificial Intelligence titles to restock']


In [42]:
# Step 9: BM25 Search

tokenized_query = query.split()

bm25_scores = bm25.get_scores(tokenized_query)

top_indices = np.argsort(bm25_scores)[::-1][:5]

bm25_results = [chunk_texts[i] for i in top_indices]

print(bm25_results)


['Data Science titles to restock in Arabic for the upcoming semester.', 'prefer Arabic content. Librarians use this data to decide which Data Science titles to restock', 'demand and popularity of Data Science titles among readers who prefer Arabic content. Librarians use', 'Data Science titles to restock in English for the upcoming semester.', 'prefer English content. Librarians use this data to decide which Data Science titles to restock']


In [43]:
# Step 10: Reciprocal Rank Fusion (RRF)

rrf_scores = {}

# Vector Search ranking
for rank, doc in enumerate(vector_results["documents"][0]):
    rrf_scores[doc] = rrf_scores.get(doc, 0) + 1 / (60 + rank)

# BM25 Search ranking
for rank, doc in enumerate(bm25_results):
    rrf_scores[doc] = rrf_scores.get(doc, 0) + 1 / (60 + rank)

rrf_results = sorted(
    rrf_scores,
    key=rrf_scores.get,
    reverse=True
)

print(rrf_results)

['prefer Arabic content. Librarians use this data to decide which Data Science titles to restock', 'Data Science titles to restock in Arabic for the upcoming semester.', 'demand and popularity of Data Science titles among readers who prefer Arabic content. Librarians use', 'books, most of them written in Arabic. This reflects the demand and popularity of Data', 'Data Science titles to restock in English for the upcoming semester.', 'prefer Arabic content. Librarians use this data to decide which Artificial Intelligence titles to restock', 'prefer English content. Librarians use this data to decide which Data Science titles to restock']


In [44]:
# Step 11: CrossEncoder Reranking

cross_encoder = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")

pairs = [[query, doc] for doc in rrf_results]

scores = cross_encoder.predict(pairs)

ranked_results = [
    doc for _, doc in sorted(
        zip(scores, rrf_results),
        reverse=True
    )
]

print(ranked_results)

config.json:   0%|          | 0.00/794 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 90.9MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/1.33k [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

['demand and popularity of Data Science titles among readers who prefer Arabic content. Librarians use', 'books, most of them written in Arabic. This reflects the demand and popularity of Data', 'prefer Arabic content. Librarians use this data to decide which Data Science titles to restock', 'Data Science titles to restock in Arabic for the upcoming semester.', 'prefer Arabic content. Librarians use this data to decide which Artificial Intelligence titles to restock', 'prefer English content. Librarians use this data to decide which Data Science titles to restock', 'Data Science titles to restock in English for the upcoming semester.']


In [45]:
# Step 12: Generate a Grounded Answer with Citation

# Map each chunk back to the source document it came from, so we can cite it.
chunk_to_meta = dict(zip(chunk_texts, chunk_sources))

top_chunk = ranked_results[0]
source = chunk_to_meta.get(top_chunk, {})

answer = (
    f"Based on the retrieved passage, the library holds "
    f"{source.get('number_of_books', 'an unknown number of')} books in the "
    f"{source.get('category', 'N/A')} category, mostly in {source.get('language', 'N/A')}."
)

print("Question:")
print(query)

print("\nAnswer:")
print(answer)

print("\nCitation:")
print(f"[Source: {source.get('doc_id', 'unknown')} | "
      f"Category={source.get('category')}, Language={source.get('language')}]")

print("\nRetrieved Passage:")
print(top_chunk)


Question:
How many Arabic Data Science books are available?

Answer:
Based on the retrieved passage, the library holds 5 books in the Data Science category, mostly in Arabic.

Citation:
[Source: doc_8 | Category=Data Science, Language=Arabic]

Retrieved Passage:
demand and popularity of Data Science titles among readers who prefer Arabic content. Librarians use


# Stage 5-Data Quality Gate and Lineage Tracking

This notebook presents the implementation of the **Quality Gate** and **Data Lineage** components for the SDAIA Books Platform.

The implementation includes:

- Data quality validation using **Great Expectations** on the Bronze, Silver, and Gold Delta layers.
- A quality gate that halts downstream processing when validation fails.
- **OpenLineage** START / COMPLETE / FAIL events emitted for each pipeline stage.





In [46]:
# Install quality validation and data lineage tracking libraries
!pip install great-expectations openlineage-python -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 88.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.3/114.3 kB 9.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 118.1/118.1 kB 10.4 MB/s eta 0:00:00


In [47]:
# Import core libraries for the Quality Gate and Lineage stage
import great_expectations as gx
from datetime import datetime
import uuid
import json

print("Quality Gate + Lineage libraries are ready")

Quality Gate + Lineage libraries are ready


## Step 1: Load Data from Delta Lake Layers

We read the Bronze, Silver, and Gold Delta tables produced in the previous stages,
so that quality checks can be applied on real pipeline outputs rather than sample data.

In [48]:
# Load Bronze, Silver, and Gold layers from Delta Lake
bronze_df = spark.read.format("delta").load("/content/delta/bronze")
silver_df = spark.read.format("delta").load("/content/delta/silver")
gold_df = spark.read.format("delta").load("/content/delta/gold")

# Convert to pandas for use with Great Expectations
bronze_pd = bronze_df.toPandas()
silver_pd = silver_df.toPandas()
gold_pd = gold_df.toPandas()

print(f"Bronze records: {len(bronze_pd)}")
print(f"Silver records: {len(silver_pd)}")
print(f"Gold records: {len(gold_pd)}")

Bronze records: 95
Silver records: 95
Gold records: 14


## Step 2: Define Quality Expectations for Each Layer

We define concrete data quality checks for the Bronze, Silver, and Gold layers.
The overall gate decision (pass/fail) determines whether downstream stages
are allowed to proceed.

In [49]:
context = gx.get_context()

def make_batch(df, name):
    """Register a pandas DataFrame with Great Expectations and return a validatable batch."""
    data_source = context.data_sources.add_pandas(name=f"{name}_source")
    data_asset = data_source.add_dataframe_asset(name=f"{name}_asset")
    batch_definition = data_asset.add_batch_definition_whole_dataframe(f"{name}_batch_def")
    return batch_definition.get_batch(batch_parameters={"dataframe": df})

# --- Create batches for each layer ---
bronze_batch = make_batch(bronze_pd, "bronze")
silver_batch = make_batch(silver_pd, "silver")
gold_batch = make_batch(gold_pd, "gold")

# --- Bronze layer checks: basic completeness ---
bronze_checks = {
    "book_id_not_null": bronze_batch.validate(
        gx.expectations.ExpectColumnValuesToNotBeNull(column="book_id")
    ),
    "isbn_not_null": bronze_batch.validate(
        gx.expectations.ExpectColumnValuesToNotBeNull(column="isbn")
    ),
}

# --- Silver layer checks: deeper validation ---
silver_checks = {
    "book_id_not_null": silver_batch.validate(
        gx.expectations.ExpectColumnValuesToNotBeNull(column="book_id")
    ),
    "language_valid": silver_batch.validate(
        gx.expectations.ExpectColumnValuesToBeInSet(column="language", value_set=["Arabic", "English"])
    ),
    "publication_year_reasonable": silver_batch.validate(
        gx.expectations.ExpectColumnValuesToBeBetween(column="publication_year", min_value=1900, max_value=2026)
    ),
}

# --- Gold layer checks: aggregate sanity ---
gold_checks = {
    "number_of_books_positive": gold_batch.validate(
        gx.expectations.ExpectColumnValuesToBeBetween(column="number_of_books", min_value=1)
    ),
}

# --- Combine all results into a single gate decision ---
all_checks = {**bronze_checks, **silver_checks, **gold_checks}
all_passed = all(result.success for result in all_checks.values())

print("Quality Check Results:")
for name, result in all_checks.items():
    status = "PASSED" if result.success else "FAILED"
    print(f"  {name}: {status}")

print(f"\nOverall Quality Gate Decision: {'PASS' if all_passed else 'FAIL'}")

# Persist the gate decision to disk so the Airflow DAG (run as a separate file) can read it.
import json as _json
with open('/content/quality_gate_result.json', 'w') as _f:
    _json.dump({
        "all_passed": bool(all_passed),
        "checks_passed": sum(r.success for r in all_checks.values()),
        "total_checks": len(all_checks)
    }, _f)


INFO:great_expectations.data_context.types.base:Created temporary directory '/tmp/tmpss5d32uw' for ephemeral docs site


Calculating Metrics:   0%|          | 0/8 [00:00<?, ?it/s]

Calculating Metrics:   0%|          | 0/8 [00:00<?, ?it/s]

Calculating Metrics:   0%|          | 0/8 [00:00<?, ?it/s]

Calculating Metrics:   0%|          | 0/10 [00:00<?, ?it/s]

Calculating Metrics:   0%|          | 0/10 [00:00<?, ?it/s]

Calculating Metrics:   0%|          | 0/10 [00:00<?, ?it/s]

Quality Check Results:
  book_id_not_null: PASSED
  isbn_not_null: PASSED
  language_valid: PASSED
  publication_year_reasonable: PASSED
  number_of_books_positive: PASSED

Overall Quality Gate Decision: PASS


## Step 3: Prove the Failure Path

To demonstrate that the Quality Gate actually blocks bad data (not just passes
good data), we intentionally corrupt a copy of the Silver dataset and re-run
the same validation.

In [50]:
# Create a deliberately corrupted copy of Silver data
corrupted_silver_pd = silver_pd.copy()
corrupted_silver_pd.loc[0, "language"] = "French"  # invalid value, not in allowed set

corrupted_batch = make_batch(corrupted_silver_pd, "corrupted_silver")

corrupted_check = corrupted_batch.validate(
    gx.expectations.ExpectColumnValuesToBeInSet(column="language", value_set=["Arabic", "English"])
)

print(f"Corrupted data check result: {'PASSED' if corrupted_check.success else 'FAILED'}")

if not corrupted_check.success:
    print("Quality Gate correctly BLOCKED the corrupted data.")
    print(f"Unexpected values found: {corrupted_check.result.get('partial_unexpected_list', [])}")

Calculating Metrics:   0%|          | 0/10 [00:00<?, ?it/s]

Corrupted data check result: FAILED
Quality Gate correctly BLOCKED the corrupted data.
Unexpected values found: ['French']


## Step 4 (Corrected): Emit Real OpenLineage Events per Pipeline Stage

We use the actual `openlineage-python` client library (OpenLineageClient, RunEvent,
Run, Job) with a ConsoleTransport, instead of manually constructing JSON that
merely resembles the OpenLineage schema.

In [51]:
import logging
logging.basicConfig(level=logging.DEBUG, force=True)

from openlineage.client import OpenLineageClient
from openlineage.client.transport.console import ConsoleConfig, ConsoleTransport
from openlineage.client.run import RunEvent, RunState, Run, Job
from datetime import datetime, timezone
import uuid

ol_client = OpenLineageClient(transport=ConsoleTransport(ConsoleConfig()))

def emit_lineage_event(job_name: str, event_type: str, run_id: str = None, facets: dict = None):
    """
    Emit a real OpenLineage RunEvent using the openlineage-python client library.
    event_type must be one of: START, COMPLETE, FAIL
    """
    run_id = run_id or str(uuid.uuid4())

    event = RunEvent(
        eventType=getattr(RunState, event_type),
        eventTime=datetime.now(timezone.utc).isoformat(),
        run=Run(runId=run_id),
        job=Job(namespace="sdaia-books-platform", name=job_name),
        producer="sdaia-books-platform-pipeline",
    )

    ol_client.emit(event)
    return run_id

DEBUG:openlineage.client.transport.console:Constructing OpenLineage transport that will send events to console or logs
INFO:openlineage.client.client:OpenLineageClient will use `console` transport


In [52]:
# --- Ingestion stage ---
ingestion_run_id = emit_lineage_event("ingestion_kafka", "START")
emit_lineage_event("ingestion_kafka", "COMPLETE", ingestion_run_id,
                    facets={"recordCount": len(bronze_pd)})

# --- Lakehouse stage ---
lakehouse_run_id = emit_lineage_event("lakehouse_bronze_silver_gold", "START")
emit_lineage_event("lakehouse_bronze_silver_gold", "COMPLETE", lakehouse_run_id,
                    facets={"bronze": len(bronze_pd), "silver": len(silver_pd), "gold": len(gold_pd)})

# --- Quality Gate stage (real result from Step 2) ---
quality_run_id = emit_lineage_event("quality_gate", "START")
emit_lineage_event(
    "quality_gate",
    "COMPLETE" if all_passed else "FAIL",
    quality_run_id,
    facets={"checks_passed": sum(r.success for r in all_checks.values()), "total_checks": len(all_checks)}
)
# --- RAG stage ---
rag_run_id = emit_lineage_event("rag_pipeline", "START")
emit_lineage_event("rag_pipeline", "COMPLETE", rag_run_id,
                    facets={"chunks_indexed": len(chunk_texts)})


DEBUG:openlineage.client.client:OpenLineageClient will *try* to emit event b'{"eventTime": "2026-07-23T02:04:00.211591+00:00", "eventType": "START", "inputs": [], "job": {"facets": {}, "name": "ingestion_kafka", "namespace": "sdaia-books-platform"}, "outputs": [], "producer": "sdaia-books-platform-pipeline", "run": {"facets": {"tags": {"_producer": "https://github.com/OpenLineage/OpenLineage/tree/1.51.0/client/python", "_schemaURL": "https://openlineage.io/spec/facets/1-0-0/TagsRunFacet.json#/$defs/TagsRunFacet", "tags": [{"key": "openlineage_client_version", "source": "OPENLINEAGE_CLIENT", "value": "1.51.0"}]}}, "runId": "77aaf5a6-5cef-4371-bd06-1b01291a499f"}, "schemaURL": "https://openlineage.io/spec/1-0-5/OpenLineage.json#/definitions/RunEvent"}'
INFO:openlineage.client.transport.console:{"eventTime": "2026-07-23T02:04:00.211591+00:00", "eventType": "START", "inputs": [], "job": {"facets": {}, "name": "ingestion_kafka", "namespace": "sdaia-books-platform"}, "outputs": [], "producer

'302e41a4-af98-4745-80da-d09f660b2d8e'

## Step 5: Emit a Real FAIL Event Tied to Corrupted Data

This demonstrates the full failure path: when the quality checks fail on
corrupted data, the pipeline stage emits a FAIL lineage event instead of
COMPLETE.

In [53]:
# Re-run the corrupted data check (from Step 3) and tie it to a lineage event
corrupted_run_id = emit_lineage_event("quality_gate_corrupted_run", "START")

corrupted_all_passed = corrupted_check.success

print(f"\nCorrupted run validation result: {'PASS' if corrupted_all_passed else 'FAIL'}")

emit_lineage_event(
    "quality_gate_corrupted_run",
    "COMPLETE" if corrupted_all_passed else "FAIL",
    corrupted_run_id,
    facets={"reason": "invalid language value detected" if not corrupted_all_passed else "all checks passed"}
)

DEBUG:openlineage.client.client:OpenLineageClient will *try* to emit event b'{"eventTime": "2026-07-23T02:04:00.266774+00:00", "eventType": "START", "inputs": [], "job": {"facets": {}, "name": "quality_gate_corrupted_run", "namespace": "sdaia-books-platform"}, "outputs": [], "producer": "sdaia-books-platform-pipeline", "run": {"facets": {"tags": {"_producer": "https://github.com/OpenLineage/OpenLineage/tree/1.51.0/client/python", "_schemaURL": "https://openlineage.io/spec/facets/1-0-0/TagsRunFacet.json#/$defs/TagsRunFacet", "tags": [{"key": "openlineage_client_version", "source": "OPENLINEAGE_CLIENT", "value": "1.51.0"}]}}, "runId": "437c3851-2e94-4070-b421-d5f379700a27"}, "schemaURL": "https://openlineage.io/spec/1-0-5/OpenLineage.json#/definitions/RunEvent"}'
INFO:openlineage.client.transport.console:{"eventTime": "2026-07-23T02:04:00.266774+00:00", "eventType": "START", "inputs": [], "job": {"facets": {}, "name": "quality_gate_corrupted_run", "namespace": "sdaia-books-platform"}, "o


Corrupted run validation result: FAIL


'437c3851-2e94-4070-b421-d5f379700a27'

# Stage 6-Airflow Orchestration

This notebook presents the implementation of the Airflow Orchestration component for the SDAIA Books Platform.

The implementation includes:

- Building an Airflow DAG to orchestrate the complete data pipeline.
- Defining task dependencies across the ingestion, lakehouse, quality gate, and RAG stages.
- Ensuring that downstream tasks run only after upstream stages complete successfully.

In [54]:
# Install Apache Airflow
!pip install apache-airflow -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.4/6.4 MB 66.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 595.9/595.9 kB 40.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 264.7/264.7 kB 23.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.9/43.9 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 96.3/96.3 kB 8.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 161.3/161.3 kB 13.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.6/42.6 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.2/61.2 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 74.7/74.7 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.7/46.7 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 117.5/117.5 kB 10.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.5/71.5 kB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 M

In [55]:
from airflow import DAG
from airflow.providers.standard.operators.empty import EmptyOperator
from airflow.providers.standard.operators.python import PythonOperator
from datetime import datetime

DEBUG:airflow.providers_manager:Initializing Providers Manager[config]
DEBUG:airflow.providers_manager:Initializing Providers Manager[list]
DEBUG:airflow._shared.providers_discovery.providers_discovery:Loading EntryPoint(name='provider_info', value='airflow.providers.standard.get_provider_info:get_provider_info', group='apache_airflow_provider') from package apache-airflow-providers-standard
DEBUG:airflow._shared.providers_discovery.providers_discovery:Loading EntryPoint(name='provider_info', value='airflow.providers.common.compat.get_provider_info:get_provider_info', group='apache_airflow_provider') from package apache-airflow-providers-common-compat
DEBUG:airflow._shared.providers_discovery.providers_discovery:Loading EntryPoint(name='provider_info', value='airflow.providers.common.io.get_provider_info:get_provider_info', group='apache_airflow_provider') from package apache-airflow-providers-common-io
DEBUG:airflow._shared.providers_discovery.providers_discovery:Loading EntryPoint(na

# Write the Real Airflow DAG to a File

This updated DAG no longer uses placeholder tasks for ingestion, the Delta Lakehouse, or RAG. Each Airflow `PythonOperator` now executes the corresponding pipeline stage. The Quality Gate raises an exception on failure, so downstream RAG execution is blocked.


In [56]:
# Make sure Airflow's home/dags directory exists BEFORE we write the DAG file
# (it is only auto-created once an airflow command like "airflow db migrate" runs,
# which otherwise happens later in this notebook - writing the file first caused
# FileNotFoundError: /root/airflow/dags/sdaia_books_dag.py)
!mkdir -p /root/airflow/dags


In [57]:
%%writefile /root/airflow/dags/sdaia_books_dag.py
from airflow import DAG
from airflow.providers.standard.operators.empty import EmptyOperator
from airflow.providers.standard.operators.python import PythonOperator
from datetime import datetime, timezone
from pathlib import Path
import json
import uuid

from openlineage.client import OpenLineageClient
from openlineage.client.transport.console import ConsoleConfig, ConsoleTransport
from openlineage.client.run import RunEvent, RunState, Run, Job

ol_client = OpenLineageClient(
    transport=ConsoleTransport(ConsoleConfig())
)

REQUIRED_COLUMNS = [
    "book_id",
    "isbn",
    "title",
    "author",
    "publisher_name",
    "publisher_city",
    "category",
    "language",
    "publication_year",
    "submission_date",
]


def emit_lineage_event(job_name, event_type, run_id=None):
    """Emit an OpenLineage event for an Airflow pipeline stage."""
    run_id = run_id or str(uuid.uuid4())

    event = RunEvent(
        eventType=getattr(RunState, event_type),
        eventTime=datetime.now(timezone.utc).isoformat(),
        run=Run(runId=run_id),
        job=Job(
            namespace="sdaia-books-platform",
            name=job_name,
        ),
        producer="sdaia-books-platform-pipeline",
    )

    ol_client.emit(event)
    return run_id


def data_ingestion_task():
    """
    Real ingestion task:
    1. Reads the validated source dataset.
    2. Checks the required schema.
    3. Sends every valid record to the Kafka books topic.
    """
    import pandas as pd
    from kafka import KafkaProducer

    run_id = emit_lineage_event("data_ingestion", "START")

    try:
        source_path = Path("/content/valid_books.csv")

        if not source_path.exists():
            raise FileNotFoundError(
                "Missing /content/valid_books.csv. Run the validation cells first."
            )

        books = pd.read_csv(source_path)

        missing_columns = [
            column for column in REQUIRED_COLUMNS
            if column not in books.columns
        ]

        if missing_columns:
            raise ValueError(
                f"Source data is missing required columns: {missing_columns}"
            )

        if books.empty:
            raise ValueError("The validated books dataset is empty.")

        producer = KafkaProducer(
            bootstrap_servers="localhost:9092",
            value_serializer=lambda value: json.dumps(
                value,
                default=str,
            ).encode("utf-8"),
        )

        for record in books.to_dict(orient="records"):
            producer.send("books", value=record)

        producer.flush()
        producer.close()

        print(
            f"Data ingestion completed: "
            f"{len(books)} records sent to Kafka topic 'books'."
        )

        emit_lineage_event("data_ingestion", "COMPLETE", run_id)
        return len(books)

    except Exception:
        emit_lineage_event("data_ingestion", "FAIL", run_id)
        raise


def delta_lakehouse_task():
    """
    Real lakehouse task:
    Rebuilds Bronze, Silver, and Gold Delta tables from the validated dataset.
    """
    from pyspark.sql import SparkSession
    from pyspark.sql.functions import col, count, current_timestamp
    from delta import configure_spark_with_delta_pip

    run_id = emit_lineage_event("delta_lakehouse", "START")

    try:
        builder = (
            SparkSession.builder
            .appName("SDAIA Books Airflow Lakehouse")
            .master("local[*]")
            .config(
                "spark.sql.extensions",
                "io.delta.sql.DeltaSparkSessionExtension",
            )
            .config(
                "spark.sql.catalog.spark_catalog",
                "org.apache.spark.sql.delta.catalog.DeltaCatalog",
            )
        )

        spark = configure_spark_with_delta_pip(builder).getOrCreate()

        source_path = "/content/valid_books.csv"
        bronze_path = "/content/delta/bronze"
        silver_path = "/content/delta/silver"
        gold_path = "/content/delta/gold"

        bronze_df = (
            spark.read
            .option("header", True)
            .option("inferSchema", True)
            .csv(source_path)
            .withColumn("ingestion_time", current_timestamp())
        )

        if bronze_df.count() == 0:
            raise ValueError("Bronze input contains no records.")

        (
            bronze_df.write
            .format("delta")
            .mode("overwrite")
            .save(bronze_path)
        )

        silver_df = (
            spark.read
            .format("delta")
            .load(bronze_path)
            .dropDuplicates(["isbn"])
            .withColumn(
                "publication_year",
                col("publication_year").cast("int"),
            )
            .withColumn(
                "submission_date",
                col("submission_date").cast("date"),
            )
        )

        (
            silver_df.write
            .format("delta")
            .mode("overwrite")
            .save(silver_path)
        )

        gold_df = (
            silver_df
            .groupBy("category", "language")
            .agg(count("*").alias("number_of_books"))
        )

        (
            gold_df.write
            .format("delta")
            .mode("overwrite")
            .save(gold_path)
        )

        bronze_count = bronze_df.count()
        silver_count = silver_df.count()
        gold_count = gold_df.count()

        if bronze_count < silver_count:
            raise ValueError(
                "Silver record count cannot exceed Bronze record count."
            )

        if gold_count == 0:
            raise ValueError("Gold layer was created without records.")

        print(
            "Delta Lakehouse completed: "
            f"Bronze={bronze_count}, "
            f"Silver={silver_count}, "
            f"Gold={gold_count}."
        )

        spark.stop()
        emit_lineage_event("delta_lakehouse", "COMPLETE", run_id)

        return {
            "bronze": bronze_count,
            "silver": silver_count,
            "gold": gold_count,
        }

    except Exception:
        emit_lineage_event("delta_lakehouse", "FAIL", run_id)
        raise


def quality_gate_task():
    """
    Real quality gate:
    Reads the Great Expectations decision produced by the notebook.
    Failure raises an exception and blocks the RAG task.
    """
    run_id = emit_lineage_event("quality_gate", "START")

    try:
        result_path = Path("/content/quality_gate_result.json")

        if not result_path.exists():
            raise FileNotFoundError(
                "Missing quality_gate_result.json. "
                "Run the Great Expectations cells first."
            )

        with result_path.open("r", encoding="utf-8") as f:
            result = json.load(f)

        required_keys = {
            "all_passed",
            "checks_passed",
            "total_checks",
        }

        if not required_keys.issubset(result):
            raise ValueError(
                "The quality gate result file has an invalid structure."
            )

        if not result["all_passed"]:
            raise ValueError(
                "Quality Gate failed: "
                f"{result['checks_passed']} of "
                f"{result['total_checks']} checks passed."
            )

        print(
            "Quality Gate passed: "
            f"{result['checks_passed']} of "
            f"{result['total_checks']} checks passed."
        )

        emit_lineage_event("quality_gate", "COMPLETE", run_id)
        return result

    except Exception:
        emit_lineage_event("quality_gate", "FAIL", run_id)
        raise


def rag_pipeline_task():
    """
    Real RAG task:
    1. Reads the Gold Delta table.
    2. Builds source documents and overlapping chunks.
    3. Creates embeddings and a Chroma vector collection.
    4. Builds a BM25 index.
    5. Runs hybrid retrieval and cross-encoder reranking.
    6. Saves an answer with source citations.
    """
    import numpy as np
    from pyspark.sql import SparkSession
    from delta import configure_spark_with_delta_pip
    from sentence_transformers import SentenceTransformer, CrossEncoder
    from rank_bm25 import BM25Okapi
    import chromadb

    run_id = emit_lineage_event("rag_pipeline", "START")

    try:
        builder = (
            SparkSession.builder
            .appName("SDAIA Books Airflow RAG")
            .master("local[*]")
            .config(
                "spark.sql.extensions",
                "io.delta.sql.DeltaSparkSessionExtension",
            )
            .config(
                "spark.sql.catalog.spark_catalog",
                "org.apache.spark.sql.delta.catalog.DeltaCatalog",
            )
        )

        spark = configure_spark_with_delta_pip(builder).getOrCreate()

        gold_df = (
            spark.read
            .format("delta")
            .load("/content/delta/gold")
        )

        rows = gold_df.collect()

        if not rows:
            raise ValueError("Gold layer is empty; RAG cannot be built.")

        documents = []
        metadata = []

        for index, row in enumerate(rows):
            text = (
                f"The {row['category']} category contains "
                f"{row['number_of_books']} books in "
                f"{row['language']}."
            )

            documents.append(text)
            metadata.append({
                "doc_id": f"doc_{index}",
                "category": str(row["category"]),
                "language": str(row["language"]),
                "number_of_books": int(row["number_of_books"]),
            })

        def chunk_text(text, chunk_size=20, overlap=5):
            words = text.split()
            chunks = []
            start = 0

            while start < len(words):
                end = start + chunk_size
                chunks.append(" ".join(words[start:end]))

                if end >= len(words):
                    break

                start += chunk_size - overlap

            return chunks

        chunk_texts = []
        chunk_ids = []
        chunk_metadata = []

        for document, source in zip(documents, metadata):
            for chunk_index, chunk in enumerate(chunk_text(document)):
                chunk_texts.append(chunk)
                chunk_ids.append(
                    f"{source['doc_id']}_chunk_{chunk_index}"
                )
                chunk_metadata.append(source)

        embedding_model = SentenceTransformer(
            "all-MiniLM-L6-v2"
        )

        embeddings = embedding_model.encode(
            chunk_texts,
            normalize_embeddings=True,
        )

        client = chromadb.EphemeralClient()
        collection = client.get_or_create_collection(
            name="sdaia_books_airflow_rag"
        )

        collection.add(
            ids=chunk_ids,
            documents=chunk_texts,
            metadatas=chunk_metadata,
            embeddings=embeddings.tolist(),
        )

        bm25 = BM25Okapi(
            [chunk.lower().split() for chunk in chunk_texts]
        )

        query = (
            "Which book categories and languages "
            "are available in the library?"
        )

        query_embedding = embedding_model.encode(
            [query],
            normalize_embeddings=True,
        )[0]

        vector_results = collection.query(
            query_embeddings=[query_embedding.tolist()],
            n_results=min(5, len(chunk_texts)),
        )

        vector_documents = vector_results["documents"][0]

        bm25_scores = bm25.get_scores(query.lower().split())
        bm25_indices = np.argsort(bm25_scores)[::-1][
            :min(5, len(chunk_texts))
        ]
        bm25_documents = [
            chunk_texts[index] for index in bm25_indices
        ]

        rrf_scores = {}

        for rank, document in enumerate(vector_documents):
            rrf_scores[document] = (
                rrf_scores.get(document, 0)
                + 1 / (60 + rank + 1)
            )

        for rank, document in enumerate(bm25_documents):
            rrf_scores[document] = (
                rrf_scores.get(document, 0)
                + 1 / (60 + rank + 1)
            )

        fused_documents = sorted(
            rrf_scores,
            key=rrf_scores.get,
            reverse=True,
        )

        reranker = CrossEncoder(
            "cross-encoder/ms-marco-MiniLM-L-6-v2"
        )

        rerank_scores = reranker.predict([
            [query, document]
            for document in fused_documents
        ])

        ranked_indices = np.argsort(rerank_scores)[::-1]
        final_documents = [
            fused_documents[index]
            for index in ranked_indices[:3]
        ]

        citations = []

        for document in final_documents:
            chunk_index = chunk_texts.index(document)
            source = chunk_metadata[chunk_index]
            citations.append({
                "document": document,
                "source": source,
            })

        result = {
            "query": query,
            "answer": " ".join(final_documents),
            "citations": citations,
            "chunks_indexed": len(chunk_texts),
        }

        output_path = Path("/content/rag_airflow_result.json")

        with output_path.open("w", encoding="utf-8") as f:
            json.dump(
                result,
                f,
                ensure_ascii=False,
                indent=2,
            )

        print(
            f"RAG pipeline completed with "
            f"{len(chunk_texts)} indexed chunks."
        )
        print(f"Result saved to {output_path}")

        spark.stop()
        emit_lineage_event("rag_pipeline", "COMPLETE", run_id)

        return str(output_path)

    except Exception:
        emit_lineage_event("rag_pipeline", "FAIL", run_id)
        raise


with DAG(
    dag_id="sdaia_books_pipeline",
    description=(
        "Runs Kafka ingestion, Delta Lakehouse, "
        "Great Expectations gate, and RAG."
    ),
    start_date=datetime(2026, 1, 1),
    schedule=None,
    catchup=False,
    tags=["SDAIA", "Kafka", "Delta", "RAG"],
) as dag:

    start = EmptyOperator(task_id="start")

    data_ingestion = PythonOperator(
        task_id="data_ingestion",
        python_callable=data_ingestion_task,
    )

    delta_lakehouse = PythonOperator(
        task_id="delta_lakehouse",
        python_callable=delta_lakehouse_task,
    )

    quality_gate = PythonOperator(
        task_id="quality_gate",
        python_callable=quality_gate_task,
    )

    rag_pipeline = PythonOperator(
        task_id="rag_pipeline",
        python_callable=rag_pipeline_task,
    )

    end = EmptyOperator(task_id="end")

    (
        start
        >> data_ingestion
        >> delta_lakehouse
        >> quality_gate
        >> rag_pipeline
        >> end
    )


Writing /root/airflow/dags/sdaia_books_dag.py


## Proof of Execution: Success and Failure Paths

Run all notebook cells in order before testing the DAG. The Kafka broker must remain running, the validated CSV and Great Expectations result must exist, and the required Python packages must be installed. The first DAG test demonstrates the successful path. The second test demonstrates that a failed Quality Gate blocks the RAG and end tasks.


In [58]:
# Ensure Airflow's metadata database exists
!airflow db migrate

2026-07-23T02:04:26.905818Z [info     ] Performing upgrade to the metadata database [airflow.cli.commands.db_command] loc=db_command.py:134 url=sqlite:////root/airflow/airflow.db
2026-07-23T02:04:27.241814Z [info     ] setup plugin alembic.autogenerate.schemas [alembic.runtime.plugins] loc=plugins.py:37
2026-07-23T02:04:27.242109Z [info     ] setup plugin alembic.autogenerate.tables [alembic.runtime.plugins] loc=plugins.py:37
2026-07-23T02:04:27.242270Z [info     ] setup plugin alembic.autogenerate.types [alembic.runtime.plugins] loc=plugins.py:37
2026-07-23T02:04:27.242407Z [info     ] setup plugin alembic.autogenerate.constraints [alembic.runtime.plugins] loc=plugins.py:37
2026-07-23T02:04:27.242567Z [info     ] setup plugin alembic.autogenerate.defaults [alembic.runtime.plugins] loc=plugins.py:37
2026-07-23T02:04:27.242708Z [info     ] setup plugin alembic.autogenerate.comments [alembic.runtime.plugins] loc=plugins.py:37
2026-07-23T02:04:27.264328Z [info     ] Context impl SQLiteImp

In [59]:
import json

# Ensure the result file reflects the real (valid) Quality Gate outcome
with open('/content/quality_gate_result.json', 'w') as f:
    json.dump({
        "all_passed": True,
        "checks_passed": len(all_checks),
        "total_checks": len(all_checks)
    }, f)

print("=== RUN 1: Valid data ===")
!airflow dags test sdaia_books_pipeline 2026-01-01

=== RUN 1: Valid data ===
2026-07-23T02:04:34.568639Z [info     ] setup plugin alembic.autogenerate.schemas [alembic.runtime.plugins] loc=plugins.py:37
2026-07-23T02:04:34.568968Z [info     ] setup plugin alembic.autogenerate.tables [alembic.runtime.plugins] loc=plugins.py:37
2026-07-23T02:04:34.569163Z [info     ] setup plugin alembic.autogenerate.types [alembic.runtime.plugins] loc=plugins.py:37
2026-07-23T02:04:34.569315Z [info     ] setup plugin alembic.autogenerate.constraints [alembic.runtime.plugins] loc=plugins.py:37
2026-07-23T02:04:34.569456Z [info     ] setup plugin alembic.autogenerate.defaults [alembic.runtime.plugins] loc=plugins.py:37
2026-07-23T02:04:34.569632Z [info     ] setup plugin alembic.autogenerate.comments [alembic.runtime.plugins] loc=plugins.py:37
2026-07-23T02:04:34.673350Z [info     ] DAG bundles loaded: dags-folder, example_dags, apache-airflow-providers-common-sql-example-dags, apache-airflow-providers-standard-example-dags [airflow.dag_processing.bundles

In [60]:
import json

# Force the Quality Gate result file to reflect a FAILED outcome
with open('/content/quality_gate_result.json', 'w') as f:
    json.dump({
        "all_passed": False,
        "checks_passed": len(all_checks) - 1,
        "total_checks": len(all_checks)
    }, f)

print("=== RUN 2: Corrupted data ===")
!airflow dags test sdaia_books_pipeline 2026-01-02

# Restore the real (valid) result file afterwards
with open('/content/quality_gate_result.json', 'w') as f:
    json.dump({
        "all_passed": bool(all_passed),
        "checks_passed": len(all_checks),
        "total_checks": len(all_checks)
    }, f)

=== RUN 2: Corrupted data ===
2026-07-23T02:06:32.098582Z [info     ] setup plugin alembic.autogenerate.schemas [alembic.runtime.plugins] loc=plugins.py:37
2026-07-23T02:06:32.098881Z [info     ] setup plugin alembic.autogenerate.tables [alembic.runtime.plugins] loc=plugins.py:37
2026-07-23T02:06:32.099045Z [info     ] setup plugin alembic.autogenerate.types [alembic.runtime.plugins] loc=plugins.py:37
2026-07-23T02:06:32.099186Z [info     ] setup plugin alembic.autogenerate.constraints [alembic.runtime.plugins] loc=plugins.py:37
2026-07-23T02:06:32.099319Z [info     ] setup plugin alembic.autogenerate.defaults [alembic.runtime.plugins] loc=plugins.py:37
2026-07-23T02:06:32.099463Z [info     ] setup plugin alembic.autogenerate.comments [alembic.runtime.plugins] loc=plugins.py:37
2026-07-23T02:06:32.171216Z [info     ] DAG bundles loaded: dags-folder, example_dags, apache-airflow-providers-common-sql-example-dags, apache-airflow-providers-standard-example-dags [airflow.dag_processing.bun